# PyNRPF v0.1.0 — Reproduce Key Numbers

Reproduces the headline detection and correction numbers reported in the
conference paper, for both the `m7_threshold` daytime threshold rule and the
`m8_xgb` two-stage classifier.

**Inputs.** `config/run.yaml`; `dataset/raw/rpf_dataset.parquet` with its
`sha256.txt` checksum sidecar.

**Outputs.** `outputs/metrics__local_dev.json` and `.yaml`, holding the dataset
summary, the train and test split, and per-model confusion counts and scores.
Also writes the fitted `xgb1_day.pkl` and `xgb2_timestamp.pkl` bundles to
`outputs/`, which notebook 04 reads.

**Runtime.** Minutes. Training the two XGBoost stages dominates.

**Prerequisites.** The archive environment from
`publication/1_conference_paper/README.md`. Run this notebook before 02, 03 or 04.

This notebook:
1. reads config from `config/run.yaml`
2. loads local `rpf_dataset.parquet` from `dataset/raw/`
3. runs validation checks via `src.validate`
4. applies time-based train/test split
5. builds day-level aggregation table
6. runs m7_threshold, m8_xgb, evaluation


In [ ]:
# ── Environment + imports ─────────────────────────────────────────────────────
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

# Make src importable
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from src.io import (
    ensure_dir,
    get,
    load_parquet,
    load_yaml,
    req,
    verify_sha256_best_effort,
    write_json,
)
from src.validate import basic_validate

print("Python:", sys.version)
print("CWD:   ", Path.cwd())
print("REPO:  ", REPO_ROOT)

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Single source of truth: config/run.yaml.  No hard-coded constants below.

CFG_PATH = REPO_ROOT / "config" / "run.yaml"
cfg = load_yaml(CFG_PATH)
print("Config loaded from:", CFG_PATH)

# -- Run settings
RUN_TAG = str(req(cfg, "run.run_tag"))
SEED    = int(req(cfg, "run.seed"))
random.seed(SEED)
np.random.seed(SEED)

# -- Paths
DATASET_PATH = (REPO_ROOT / str(req(cfg, "paths.dataset_parquet"))).resolve()
SHA_PATH     = (REPO_ROOT / str(req(cfg, "paths.sha256_file"))).resolve()
OUTPUT_DIR   = (REPO_ROOT / str(req(cfg, "paths.output_dir"))).resolve()
ensure_dir(OUTPUT_DIR)

# -- Column names
COL_SITE  = str(req(cfg, "data.columns.site"))
COL_TS    = str(req(cfg, "data.columns.ts"))
COL_NET   = str(req(cfg, "data.columns.net_load"))
COL_SOLAR = str(req(cfg, "data.columns.solar"))
COL_GT    = str(req(cfg, "data.columns.gt"))
ALL_COLS  = [COL_SITE, COL_TS, COL_NET, COL_SOLAR, COL_GT]

# -- Data settings
INTERVAL_MINUTES = int(req(cfg, "data.interval_minutes"))
MWH_FACTOR       = float(req(cfg, "data.mwh_factor"))

# -- Validation flags
VERIFY_SHA256           = bool(get(cfg, "validation.verify_sha256_best_effort", True))
STRIP_TIMEZONE          = bool(get(cfg, "validation.strip_timezone", True))
ENFORCE_INTERVAL_ALIGN  = bool(get(cfg, "validation.enforce_interval_alignment", True))
ENFORCE_UNIQUE_KEYS     = bool(get(cfg, "validation.enforce_unique_keys", True))

# -- Train / test split dates
TRAIN_START = str(req(cfg, "split.train_start"))
TRAIN_END   = str(req(cfg, "split.train_end"))
TEST_START  = str(req(cfg, "split.test_start"))
TEST_END    = str(req(cfg, "split.test_end"))

print(f"RUN_TAG:  {RUN_TAG}")
print(f"SEED:     {SEED}")
print(f"DATASET:  {DATASET_PATH}")
print(f"COLUMNS:  {ALL_COLS}")
print(f"SPLIT:    train {TRAIN_START}..{TRAIN_END} | test {TEST_START}..{TEST_END}")

In [ ]:
# ── Ensure local dataset exists ───────────────────────────────────────────────
if not DATASET_PATH.exists():
    print("Dataset not found locally.")
    print("Please place the parquet at:", DATASET_PATH)
    raise SystemExit("Stopping: no local dataset available.")

local_path = DATASET_PATH
print("Parquet found locally:", local_path)

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
if VERIFY_SHA256:
    sha_result = verify_sha256_best_effort(local_path, SHA_PATH)
    print("SHA-256 check:", sha_result["status"],
          f"({sha_result.get('note', '')})" if sha_result.get("note") else "")

df = load_parquet(local_path)
print(df.dtypes)
df.head()

In [ ]:
# ── Validation ────────────────────────────────────────────────────────────────
result = basic_validate(
    df,
    cols_required=ALL_COLS,
    site_col=COL_SITE,
    ts_col=COL_TS,
    key_cols=[COL_SITE, COL_TS],
    interval_minutes=INTERVAL_MINUTES,
    strip_timezone=STRIP_TIMEZONE,
    enforce_interval_alignment=ENFORCE_INTERVAL_ALIGN,
    enforce_unique_keys=ENFORCE_UNIQUE_KEYS,
)

df = result["df"]
summary = result["summary"]

print("Validation passed.")
for k, v in summary.items():
    print(f"  {k}: {v}")

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
df["date"] = df[COL_TS].dt.date

train_mask = (df["date"] >= pd.Timestamp(TRAIN_START).date()) & (df["date"] <= pd.Timestamp(TRAIN_END).date())
test_mask  = (df["date"] >= pd.Timestamp(TEST_START).date())  & (df["date"] <= pd.Timestamp(TEST_END).date())

df_train = df.loc[train_mask].copy()
df_test  = df.loc[test_mask].copy()

print(f"Train: {len(df_train):,} rows  ({TRAIN_START} to {TRAIN_END})")
print(f"Test:  {len(df_test):,} rows  ({TEST_START} to {TEST_END})")
print(f"Other: {(~train_mask & ~test_mask).sum():,} rows outside split range")

In [ ]:
# ── Day-level aggregation ─────────────────────────────────────────────────────
day_df = (
    df.groupby([COL_SITE, "date"])
    .agg(
        n_intervals=(COL_TS, "count"),
        any_gt_neg=(COL_GT, lambda s: int((s < 0).any())),
    )
    .reset_index()
)

print(f"Day-level rows: {len(day_df):,}")
print(f"  Sites: {day_df[COL_SITE].nunique()}")
print(f"  Days with RPF (ground truth): {day_df['any_gt_neg'].sum():,} / {len(day_df):,}")
day_df.head()

In [ ]:
# ── m7_threshold ──────────────────────────────────────────────────────────────
# Deterministic detection + correction method (DTR).
# Config: cfg["m7_threshold"]

from src.m7_threshold import run_m7

df = run_m7(df, cfg, COL_SITE, COL_TS, COL_NET, COL_SOLAR)

# Per-site summary
m7_site = (
    df.groupby(COL_SITE)
    .agg(
        intervals_flagged=("m7_rpf_flag", "sum"),
        days_flagged=("m7_rpf_day", lambda s: s.any()),  # just a check
    )
)
print("\nPer-site m7 flagged intervals:")
print(m7_site.to_string())

# Update day_df with m7 day-level predictions
day_m7 = df.groupby([COL_SITE, "date"])["m7_rpf_flag"].any().reset_index(name="m7_rpf_day")
day_df = day_df.merge(day_m7, on=[COL_SITE, "date"], how="left")
day_df["m7_rpf_day"] = day_df["m7_rpf_day"].fillna(False)

print(f"\nDay-level: {int(day_df['m7_rpf_day'].sum()):,} days flagged by m7 "
      f"(ground truth: {int(day_df['any_gt_neg'].sum()):,})")

In [ ]:
# ── m8_xgb ────────────────────────────────────────────────────────────────────
# XGBoost day + timestamp classification.
# Config: cfg["m8_xgb"]["xgb1_day"], cfg["m8_xgb"]["xgb2_timestamp"]
from src.m8_xgb import run_m8

df = run_m8(df, cfg, COL_SITE, COL_TS, COL_NET, COL_SOLAR, COL_GT)

# Day-level confusion on test set
day_m8 = df.loc[test_mask].groupby([COL_SITE, "date"]).agg(
    m8_pred=("m8_rpf_day", "first"),
    min_gt=(COL_GT, "min"),
).reset_index()
day_m8["y_true"] = day_m8["min_gt"] < 0
tp = ((day_m8.m8_pred) & (day_m8.y_true)).sum()
fp = ((day_m8.m8_pred) & (~day_m8.y_true)).sum()
fn = ((~day_m8.m8_pred) & (day_m8.y_true)).sum()
tn = ((~day_m8.m8_pred) & (~day_m8.y_true)).sum()
P = tp/(tp+fp) if (tp+fp) else 0
R = tp/(tp+fn) if (tp+fn) else 0
F1 = 2*P*R/(P+R) if (P+R) else 0
print("\n=== m8_xgb Day-Level TEST SET ===")
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"  P={P:.3f}  R={R:.3f}  F1={F1:.3f}")
print("  Target: F1 > 0.95")

In [ ]:
# ── Evaluation ─────────────────────────────────────────────────────────────────
# Ground truth (COL_GT) is used ONLY here for scoring — never in m7/m8 logic.

from src.evaluate import evaluate_method

eval_rounding = int(cfg.get("evaluation", {}).get("rounding", 3))
split_cfg = {"train_end": TRAIN_END, "test_start": TEST_START}

# m7_threshold evaluation
m7_results = evaluate_method(
    df, COL_SITE, COL_TS, COL_NET, COL_GT,
    pred_day_col="m7_rpf_day",
    pred_flag_col="m7_rpf_flag",
    split_cfg=split_cfg,
    method_name="m7_threshold (DTR)",
    rounding=eval_rounding,
)

# m8_xgb evaluation
m8_results = evaluate_method(
    df, COL_SITE, COL_TS, COL_NET, COL_GT,
    pred_day_col="m8_rpf_day",
    pred_flag_col="m8_rpf_flag",
    split_cfg=split_cfg,
    method_name="m8_xgb (XGBoost)",
    rounding=eval_rounding,
)

In [ ]:
# ── Export metrics JSON + YAML ─────────────────────────────────────────────────
from src.io import write_yaml

METRICS_JSON = OUTPUT_DIR / f"metrics__{RUN_TAG}.json"
METRICS_YAML = OUTPUT_DIR / f"metrics__{RUN_TAG}.yaml"

metrics = {
    "run_tag": RUN_TAG,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": str(local_path.relative_to(REPO_ROOT)),
    "split": {
        "train_start": TRAIN_START,
        "train_end": TRAIN_END,
        "test_start": TEST_START,
        "test_end": TEST_END,
    },
    "data_summary": {
        "n_rows": summary["n_rows"],
        "n_sites": summary["n_sites"],
        "n_train_rows": len(df_train),
        "n_test_rows": len(df_test),
        "n_day_rows": len(day_df),
        **{k: v for k, v in summary.items() if k.startswith("null_")},
    },
    "m7_threshold": m7_results,
    "m8_xgb": m8_results,
}

write_json(METRICS_JSON, metrics)
write_yaml(METRICS_YAML, metrics)

print(f"Wrote: {METRICS_JSON}")
print(f"Wrote: {METRICS_YAML}")
print("\n=== Done ===")